## IE Results Viewer
Viewer for IE results on OASIS journal reports. Facilitates interactive adjustment of section and significance scores to affect results ranking.

In [1]:
%%capture
import warnings
# suppress user warnings during execution
warnings.filterwarnings(action='ignore', category=UserWarning)
warnings.filterwarnings(action='ignore', category=FutureWarning)

# load required dependencies
%pip install --upgrade pip
%pip install requirements.txt

In [3]:
import pandas as pd  # for DataFrame
from ipywidgets import Dropdown, Button, Text, Output, FloatSlider, Layout, HBox, VBox, FileUpload
from IPython.display import display, HTML
from html import escape
import io
#from scipy import io  # for writing escaped HTML

# default initial values for sliders (used for reset button and as fallback if sliders are not set)
DEFAULT_SCORES: dict[str, float] = {
    "title": 40.0,              # score for terms apprearing in the title section
    "abstract": 2.0,            # score for terms apprearing in the abstract section
    "body": 0.1,                # score for terms apprearing in the body section
    "sig_proximity": 2.0,       # score for terms with significance proximity to the target term
    "mrs": 1.0,                 # Minimum Relevance Score
    "mrc": 5.0                  # Minimum Relevance Count
}


def get_input_data(filepath_or_buffer) -> list[dict]:
    df = pd.read_csv(filepath_or_buffer, delimiter=",", encoding="utf-8", skip_blank_lines=True)
    # set any NaN values to blank string
    df.fillna("", inplace=True)
    # returning data as list[dict]
    return df.to_dict(orient='records')
    

# get maximum section score from a list of sections
# (e.g. concept may be within "abstract", "page" AND "body" sections)
def get_max_section_score(sections: list[str]) -> float:    
    
    def get_section_score(section: str="") -> float:
        score_by_section: dict[float] = {
            "title": float(slider_t.value),
            "abstract": float(slider_a.value),
            "body": float(slider_b.value),
            "end_matter": 0.0
        }
        return score_by_section.get(section.strip().lower(), 0.0)        

    return max([get_section_score(sec) for sec in sections], default=0.0)
   

# create HTML anchor for displaying id that looks like URL
def make_html_link(id, lbl):
    label = escape(lbl)
    if id.startswith("http"):
        return f"<a target='_blank' rel='noopener noreferrer' href='{id}'>{label}</a>"
    else:
        return label


# calculate aggregated scores per concept id           
def aggregate_results_by_concept(data: list[dict]) -> pd.DataFrame:
    sum_by_id = {}
    
    for row in data:
        # create list from csv delimited string of sections
        sections = list(row.get('sections', '').split(','))
        # if item is located in the end matter don't use it
        if('end_matter' in sections):
            continue
        
        # for concept_id use 'id' field, or text field if not present 
        concept_id: str = row.get('id', None)
        if(concept_id or '') == '':
            concept_id = row.get('text', 'n/a').lower()

        # do we have a record for this concepot?        
        if concept_id not in sum_by_id.keys():
            # create new record for this concept
            sum_by_id[concept_id] = {
                'id': concept_id,
                'text': [],
                'label': row.get('label', ''),
                'sec_score': 0.0,
                'sig_score': 0.0,
                'score': 0.0,
                'count': 0
            }
        new_section_score: float = get_max_section_score(sections)
        new_sig_proximity: float = round(slider_s.value, 2) if row.get('sig_proximity', 0) > 0 else 0.0
        new_concept_text: str = row.get('text', '')
        if new_concept_text != "" and new_concept_text not in sum_by_id[concept_id]['text']:
            sum_by_id[concept_id]['text'].append(new_concept_text)
        # aggregate scores for this concept
        sum_by_id[concept_id]['sec_score'] += new_section_score
        sum_by_id[concept_id]['sig_score'] += new_sig_proximity
        sum_by_id[concept_id]['score'] += new_section_score + new_sig_proximity        
        sum_by_id[concept_id]['count'] += 1
           
    # return as a DataFrame, adding combined columns for display purposes
    df = pd.DataFrame(list(sum_by_id.values()))
    df["span"] = df.apply(lambda x: make_html_link(x['id'], ", ".join(x['text'])), axis=1)
    df["score_explain"] = df.apply(lambda x: f"({round(x['sec_score'], 2)} + {round(x['sig_score'], 2)})", axis=1)
    return df


# UI component for input data file selection
file_input = FileUpload(
    button_style='primary',
    description="Select CSV file",  # Button text
    accept='.csv',  # Accepted file extension e.g. '.txt', '.pdf', 'image/*', 'image/*,.pdf'
    multiple=False,  # True to accept multiple files upload else False
    layout=Layout(width='200px')
)

# when a file is selected display the selected file name 
file_name_display: Output = Output()
def on_upload_change(change):
    file_name_display.clear_output()
    s = ""
    for filename in file_input.value:
        s += f" Selected: {filename.get('name', '-')}"
    with file_name_display:
        display(HTML(s))
file_input.observe(on_upload_change, names='value')

# UI slider controls
s_style = {'description_width': '150px', 'handle_color': 'lightblue'}
s_layout=Layout(width='500px')
slider_t = FloatSlider(description='Title score', value=DEFAULT_SCORES.get("title", 0.0), min=0.0, max=50.0, step=0.1, layout=s_layout, style=s_style, tooltip="Score for terms appearing in the title section")
slider_a = FloatSlider(description='Abstract score', value=DEFAULT_SCORES.get("abstract", 0.0), min=0.0, max=50.0, step=0.1, layout=s_layout, style=s_style, tooltip="Score for terms appearing in the abstract section")
slider_b = FloatSlider(description='Body score', value=DEFAULT_SCORES.get("body", 0.0), min=0.0, max=50.0, step=0.1, layout=s_layout, style=s_style, tooltip="Score for terms appearing in the body section")
slider_s = FloatSlider(description='Significance score', value=DEFAULT_SCORES.get("sig_proximity", 0.0), min=0.0, max=50.0, step=0.1, layout=s_layout, style=s_style, tooltip="Score for terms close to significance terms")
slider_mrs = FloatSlider(description='Minimum score', value=DEFAULT_SCORES.get("mrs", 0.0), min=0.0, max=50.0, step=0.1, layout=s_layout, style=s_style, tooltip="Minimum relevance score for a term to be included in the results")
slider_mrc = FloatSlider(description='Minimum count', value=DEFAULT_SCORES.get("mrc", 1), min=1, max=50, step=1, layout=s_layout, style=s_style, tooltip="Minimum relevance count for a term to be included in the results")
sliders = VBox([slider_t, slider_a, slider_b, slider_s, slider_mrs, slider_mrc], layout=Layout(display='flex', flex_flow='column', gap='5px'))
# UI control buttons
b_layout = Layout(width='200px')
refresh = Button(description="Refresh results", icon='refresh', button_style='primary', layout=b_layout)
reset = Button(description="Clear and reset", icon='times', button_style='primary', layout=b_layout)
buttons = HBox([refresh, reset], layout=Layout(display='flex', flex_flow='row', gap='10px'))
# UI output for results table
outputs = Output()

# display all UI components
display(HBox([file_input, file_name_display]))
display(sliders, buttons, outputs)

# refresh results when refresh button is clicked
def on_refresh_click(b):
    outputs.clear_output()
    input_file = file_input.value[0] if file_input.value else None
    if input_file:
        input_data = get_input_data(io.BytesIO(input_file['content']))       
        
        df = aggregate_results_by_concept(input_data) # aggregated results by concept id
        df = df[df['score'] >= slider_mrs.value]  # filtering by minimum relevance score
        df = df[df['count'] >= slider_mrc.value]  # filtering by minimum relevance count
        df = df.sort_values(by='score', ascending=False) #.head(20)
        df = df[['span', 'label', 'count', 'sec_score', 'sig_score', 'score']]
        styled_df = (df.style
            .set_caption("Results aggregated by concept")
            #.hide(subset=['id', 'text'], axis=1)
            .hide(axis="index")
            .highlight_max(subset=['sig_score', 'count']) #, color='red'
            .background_gradient(subset=['score', 'sec_score', 'sig_score', 'count']) #, cmap='YlOrRd', low=0.2, high=0.8
             #.applymap(color_negative_red)
            .format(na_rep="n/a")
            .format( "{:.2f}", subset=['sec_score', 'sig_score', 'score']))
       
        with outputs:
            display(HTML(styled_df.to_html(
                index=False, 
                #border=1,
                #justify='right',
                #render_links=True, 
                #escape=False, 
                #columns=['span', 'label', 'count', 'sec_score', 'sig_score', 'score']
            )))   
                    
refresh.on_click(on_refresh_click)

# clear fields and reset sliders when reset button is clicked
def on_reset_click(b):
    slider_t.value = DEFAULT_SCORES["title"]
    slider_a.value = DEFAULT_SCORES["abstract"]
    slider_b.value = DEFAULT_SCORES["body"]
    slider_s.value = DEFAULT_SCORES["sig_proximity"]
    slider_mrs.value = DEFAULT_SCORES["mrs"]
    slider_mrc.value = DEFAULT_SCORES["mrc"]
    outputs.clear_output()
    file_input.value = []
    file_name_display.clear_output()
reset.on_click(on_reset_click)


Output()